**House Prices: Advanced Regression Techniques - Previsor de preços de imóveis**

*Este notebook contém um modelo baseado nos prática códigos do livro "Hands-On Machine Learning". Exploramos a Regressão Linear e o K-Nearest Neighbors (KNN) na prática para prever o valor de um imóvel com base em um dataset baixado do Kaggle.*

*Fique à vontade para analisar e copiar o código neste notebook!*


# Configuração


In [34]:
# Importa o módulo sys para acessar informações do Python em uso.
import sys

# Importa o scikit-learn, biblioteca usada nos modelos de machine learning.
import sklearn


# Carregando os Dados

Nesse modelo, usaremos os dados extraídos do arquivo .zip


In [35]:
# Importa Path para trabalhar com caminhos de arquivos de forma mais segura.
from pathlib import Path

# Importa pandas para ler e manipular os dados em formato de tabela.
import pandas as pd

# Define o caminho onde os arquivos do dataset foram extraídos.
DATASET_DIR = Path("datasets/house-prices")

# Verifica se esse caminho existe a partir da pasta atual do notebook.
if not DATASET_DIR.exists():
    # Se não existir, usa o caminho considerando a raiz do projeto Hands-on.
    DATASET_DIR = Path("Projetos-ML/datasets/house-prices")


# Cria uma função para carregar os dados de treino.
def carregar_dados_houses():
    # Lê o arquivo train.csv e devolve um DataFrame do pandas.
    return pd.read_csv(DATASET_DIR / "train.csv")


# Carrega os dados em uma variável chamada housing.
housing = carregar_dados_houses()


## Analisar a estrutura de dados do dataset

In [36]:
# Mostra as 5 primeiras linhas do dataset.
housing.head()


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [37]:
# Mostra o tipo de cada coluna e a quantidade de valores preenchidos.
housing.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [38]:
# Mostra estatísticas básicas das colunas numéricas.
housing.describe()


,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


# Pré-processamento dos dados

Usando apenas colunas numericas e tendo `SalePrice` como alvo.


In [39]:
# Importa a função que separa os dados em treino e teste.
from sklearn.model_selection import train_test_split

# Importa o padronizador que coloca as colunas na mesma escala.
from sklearn.preprocessing import StandardScaler


In [ ]:
# Lista com as colunas que serão usadas para treinar o modelo.
# Foram escolhidas colunas numéricas e sem valores ausentes.
colunas_modelo = [
    # Tipo/classe do imóvel.
    "MSSubClass",
    # Tamanho do terreno.
    "LotArea",
    # Qualidade e condição geral do imóvel.
    "OverallQual",
    "OverallCond",
    # Ano de construção e reforma.
    "YearBuilt",
    "YearRemodAdd",
    # Informações do porão.
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    # Áreas dos andares e área habitável.
    "1stFlrSF",
    "2ndFlrSF",
    "LowQualFinSF",
    "GrLivArea",
    # Banheiros.
    "BsmtFullBath",
    "BsmtHalfBath",
    "FullBath",
    "HalfBath",
    # Cômodos.
    "BedroomAbvGr",
    "KitchenAbvGr",
    "TotRmsAbvGrd",
    # Lareiras e garagem.
    "Fireplaces",
    "GarageCars",
    "GarageArea",
    # Áreas externas.
    "WoodDeckSF",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "PoolArea",
    # Valor de itens extras e data da venda.
    "MiscVal",
    "MoSold",
    "YrSold",
]

# Define a coluna que o modelo tentará prever.
alvo = "SalePrice"

# Cria uma nova tabela apenas com as colunas escolhidas e o alvo.
dados_modelo = housing[colunas_modelo + [alvo]].copy()

# Mostra as primeiras linhas da tabela filtrada.
dados_modelo.head()


,MSSubClass,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
0,60,8450,7,5,2003,2003,706,0,150,856,...,0,61,0,0,0,0,0,2,2008,208500
1,20,9600,6,8,1976,1976,978,0,284,1262,...,298,0,0,0,0,0,0,5,2007,181500
2,60,11250,7,5,2001,2002,486,0,434,920,...,0,42,0,0,0,0,0,9,2008,223500
3,70,9550,7,5,1915,1970,216,0,540,756,...,0,35,272,0,0,0,0,2,2006,140000
4,60,14260,8,5,2000,2000,655,0,490,1145,...,192,84,0,0,0,0,0,12,2008,250000


In [41]:
# Conta valores ausentes por coluna, ordena do maior para o menor e mostra os primeiros.
dados_modelo.isna().sum().sort_values(ascending=False).head()


MSSubClass     0
LotArea        0
OverallQual    0
OverallCond    0
YearBuilt      0
dtype: int64

In [42]:
# Cria X com as colunas usadas como entrada do modelo.
X = dados_modelo[colunas_modelo].astype(float)

# Cria y com a coluna que queremos prever.
y = dados_modelo[alvo].astype(float)

# Separa os dados em treino e teste.
X_train, X_test, y_train, y_test = train_test_split(
    # Dados de entrada.
    X,
    # Valores reais que queremos prever.
    y,
    # Usa 20% dos dados para teste.
    test_size=0.2,
    # Mantém a mesma divisão sempre que o código rodar.
    random_state=42,
)

# Mostra o tamanho de cada conjunto criado.
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((1168, 33), (292, 33), (1168,), (292,))

In [43]:
# Cria o objeto responsável por padronizar os dados.
scaler = StandardScaler()

# Aprende a escala com os dados de treino e transforma esses dados.
X_train_preparado = scaler.fit_transform(X_train)

# Usa a mesma escala aprendida no treino para transformar os dados de teste.
X_test_preparado = scaler.transform(X_test)

# Mostra as 5 primeiras linhas dos dados de treino já preparados.
X_train_preparado[:5]


array([[-0.8667643 , -0.21289571, -0.82044456,  0.3722173 , -0.45546896,
        -1.34606303,  1.03726861, -0.28550406, -0.40028165,  0.57261219,
         0.37423523, -0.80192292, -0.11899866, -0.40709315,  1.10531958,
        -0.24287002, -1.05556573, -0.76409752,  0.13621832, -0.21275711,
        -0.96456591, -0.95859215, -1.05654384, -0.86383727,  1.18840233,
        -0.71435182, -0.35192107, -0.12100805, -0.27583782, -0.07099284,
        -0.09274033, -0.13341669,  1.65006527],
       [ 0.07410996, -0.26524463, -0.08893368,  1.26860866,  0.71860895,
         0.43921379, -0.97199573, -0.28550406,  0.51191969, -0.59654659,
        -0.95820221,  0.9550877 , -0.11899866,  0.08317013, -0.81869424,
        -0.24287002,  0.7736639 ,  1.23694711,  0.13621832, -0.21275711,
         0.27075534,  0.59214972,  0.29509165, -0.45626397, -0.74015748,
        -0.13801492, -0.35192107, -0.12100805, -0.27583782, -0.07099284,
        -0.09274033, -0.5080097 ,  0.89367742],
       [-0.63154574, -0.1778

Agora use `X_train_preparado`, `X_test_preparado`, `y_train` e `y_test` para treinar e avaliar os modelos.
